# CUTEst

In [ ]:
import os
from typing import Any

from pathlib import Path
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from data.CUTEst.check_CUTEst_problems import problemsToRun
from qnlab.experiment.profile import draw_pp
from qnlab.util.method import get_methods
from qnlab.experiment.for_cutest_run import run, load_results

In [ ]:
os.chdir(Path(os.path.abspath("cutest.ipynb")).parent.parent.resolve())
print(os.getcwd())

In [ ]:
def display_results(
    alg_names: list[str],
    callsM: np.ndarray,
    problems: list[str],
) -> Any:
    # Create dataframe
    data = {}
    data["problem"] = problems
    for i, alg_name in enumerate(alg_names):
        data[f"{alg_name}"] = callsM[i, :].tolist()
    df = pd.DataFrame(data)
    df.set_index("problem", inplace=True)

    # Apply styling
    def color_scale_with_cmap(row):
        if np.all(np.isinf(row.values)):
            return ["background-color: rgba(0, 0, 0, 0.8)" for _ in row.values]
        norm = plt.Normalize(vmin=row.min(), vmax=row.min() * 10)  # type:ignore
        cmap = matplotlib.colormaps["coolwarm"]
        return [
            f"background-color: rgba({int(r * 255)}, {int(g * 255)}, {int(b * 255)}, 0.8)"
            for r, g, b, _ in cmap(norm(row.values))
        ]

    styled_df = df.style.apply(color_scale_with_cmap, axis=1)

    return styled_df


In [ ]:
TOO_LONG_TIME_PROBLEMS = [
    "DMN15103LS",
    "DMN15332LS",
    "DMN37142LS",
    "DMN37143LS",
    "EIGENALS",
    "EIGENBLS",
    "EIGENCLS",
]


def generate_title(precision: int, noise: np.float64, gtol: float) -> str | None:
    """Generate title string for the performance profile plot."""
    if noise == 0:
        return None

    noise_e = int(np.log10(noise))
    assert np.isclose(noise, 10**noise_e)
    gtol_e = int(np.log10(gtol))
    assert np.isclose(gtol, 10**gtol_e)
    return (
        rf"noise=$10^{{{noise_e}}}$, "
        + r"$\epsilon_{\mathrm{gtol}}="
        + f"10^{{{gtol_e}}}$"
    )


def generate_output_path(precision: int, noise: np.float64, gtol: float) -> Path:
    """Generate output file path for the performance profile plot."""
    output_dir = Path("doc/imgs/compare")
    precision_noise = f"precision{precision}" if noise == 0 else f"noise{noise}"
    gtol_filename = f"{gtol:.0e}".replace("+", "")
    return output_dir / f"_pp_{precision_noise}_gtol{gtol_filename}.pdf"


def generate_fig_size(noise: np.float64) -> tuple[float, float]:
    """Generate figure size based on noise level."""
    return (7, 5.5) if noise > 0 else (7, 5)


def main():
    for precision, noise in [
        (64, np.float64(0.0)),
        (32, np.float64(0.0)),
        (16, np.float64(0.0)),
        (64, np.float64(1e-3)),
    ]:
        problems = problemsToRun(precision)
        problems = [p for p in problems if p not in TOO_LONG_TIME_PROBLEMS]
        methods, *_ = get_methods()
        if noise > 0:
            new_methods = []
            for method, option in methods:
                new_option = option.copy()
                new_option["gtol"] = noise * 10
                new_methods.append((method, new_option))
            methods = new_methods
        run(problems, methods, precision, noise, TL=10000)

    methods, ALGORITHM_COLORS, ALGORITHM_LINE_STYLES = get_methods()
    for precision, noise, gtol in [
        # (64, np.float64(0.0), 1e-3),
        # (64, np.float64(0.0), 1e-4),
        # (64, np.float64(0.0), 1e-5),
        # (32, np.float64(0.0), 1e-3),
        # (32, np.float64(0.0), 1e-4),
        # (32, np.float64(0.0), 1e-5),
        # (16, np.float64(0.0), 1e-3),
        # (16, np.float64(0.0), 1e-4),
        # (16, np.float64(0.0), 1e-5),
        (64, np.float64(1e-3), 1e-2),
    ]:
        problems = problemsToRun(precision)
        problems = [p for p in problems if p not in TOO_LONG_TIME_PROBLEMS]
        # individual_plot(problems, methods, precision, noise)
        alg_names, callsM, fxsM, gnormsM, problems = load_results(
            methods, problems, precision, noise, gtol=gtol
        )

        styled_df = display_results(alg_names, callsM, problems)
        if False:
            display(styled_df)

        # Generate plot parameters
        output_path = generate_output_path(precision, noise, gtol)
        fig_size = generate_fig_size(noise)
        title = generate_title(precision, noise, gtol)

        # Draw performance profile
        draw_pp(
            alg_names,
            callsM,
            ALGORITHM_COLORS,
            ALGORITHM_LINE_STYLES,
            output_path=output_path,
            fig_size=fig_size,
            title=title,
        )


In [ ]:
main()